## DNS Diagnosis — `nslookup` and `dig`

**Support context:** `ping` and `traceroute` prove whether a network *path* works. Neither one tells you anything about **name resolution** — the **step before any of that even happens**, where a hostname like `api.internal.example.com` gets turned into an IP address. A huge slice of "it's down!" tickets are actually DNS problems wearing a networking costume: the app server is fine, the network path is fine, but the client **never got the right IP** to begin with.

**Developer bridge:** DNS is the lookup table your OS/browser hits before it can even open a socket — same role as a service registry or `/etc/hosts`-style mapping you'd use in a microservices setup, except it's a distributed, cached, hierarchical system instead of a local file.
> `nslookup` and `dig` are both "query this lookup table directly and show me the raw record" tools — think of them like hitting a service registry's API endpoint by hand instead of trusting your app's built-in client.

`nslookup` is the older, simpler, interactive-friendly tool. `dig` is the modern, more powerful, more precise one — support and network engineers reach for `dig` by default once they're past the basics, but `nslookup` is still everywhere (including Windows) so you need both.

### `nslookup` — Quick, Simple DNS Query

**COMMAND:** `nslookup`

**PURPOSE (support):** Fastest way to answer **"does this hostname resolve, and to what?"** It's on nearly every OS (Linux, Windows, macOS) with near-identical output, which makes it the lowest-common-denominator tool when you're remoted into a machine you don't control and don't know what's installed.

**SYNTAX:** `nslookup [hostname] [dns-server]`

**FLAGS / USAGE:**

|Form|Meaning|
|---|---|
|`nslookup host`|query using the system's configured DNS server (from `/etc/resolv.conf`)|
|`nslookup host 8.8.8.8`|query a *specific* DNS server directly, bypassing your configured one|
|`nslookup -type=MX host`|query a specific record type (MX, NS, TXT, etc. — default is A)|

**EXAMPLE:**

In [ ]:
$ nslookup google.com
Server:         127.0.0.53
Address:        127.0.0.53#53

Non-authoritative answer:
Name:   google.com
Address: 142.250.183.14
Name:   google.com
Address: 2404:6800:4009:825::200e

**READ THE OUTPUT:**

- **`Server` / `Address` (top)** — this is *which DNS server answered your query*, not the target. `127.0.0.53` is a **local stub resolver** (systemd-resolved) — very common on modern Ubuntu/WSL, it forwards your query to a real upstream resolver behind the scenes.
- **`Non-authoritative answer`** — means the response came from a resolver's **cache**, not the domain's actual authoritative nameserver. This is normal and expected almost always; an *authoritative* answer only happens if you query the domain's own nameserver directly.
- **`Name` / `Address`** — the actual resolution result. A domain can (and often does) have multiple `A` records — that's normal load-balancing/redundancy, not an error.
- **No `Address` line at all, or `NXDOMAIN`** — the **name does not exist** as far as that DNS server is concerned. Different failure from a timeout — see GOTCHA below.

**TRY NOW:**
```
nslookup google.com
nslookup google.com 8.8.8.8
```

Compare the two — same domain, but the second one skips your local/ISP resolver entirely and asks Google's public DNS directly.

> **If the first fails but the second works, your configured DNS server is the problem, not the domain**.

**GOTCHA:** `nslookup` is officially considered **deprecated in favor of `dig`/`host`** by the maintainers (it has some parsing quirks and inconsistent behavior across platforms), but you'll still see it constantly in the wild, especially from Windows-background colleagues — don't be surprised when a ticket or runbook uses it instead of `dig`.

> **Configured DNS server**: a blank/hanging response (not `NXDOMAIN`, just silence) usually means the DNS server itself is unreachable — that's a network/firewall problem to the DNS server, not a "domain doesn't exist" problem. Check `ping <dns-server-ip>` next.

### `dig` — The Detailed DNS Tool

**COMMAND:** `dig`

**PURPOSE (support):** `dig` gives you the *raw* DNS protocol response — every field, every flag, every section — which is exactly what you want once "does it resolve?" turns into "why is it resolving *wrong*, or *slowly*, or *inconsistently*?"

> It's the tool for actually **diagnosing a DNS problem**, not just confirming one exists.

**Developer bridge:** if `nslookup` is like `curl` with no flags, `dig` is like `curl -v` — same underlying protocol, but it shows you the full request/response instead of just the final answer.

**SYNTAX:**:
```shell
dig [@server] [options] [name] [type] [class] [query-options]
dig @server name type
dig -
```
TCP/IP command. The `dig` command is used to query DNS servers; it is more flexible than the deprecated `nslookup` command. When invoked with just the `-h`option, it displays a list of options for the command. If you use it without any options or arguments, it will search for the root server. The standard arguments are:
- `server`: The server to query. If no server is supplied, `dig` will check the nameservers listed in **`/etc/resolv.conf`**. The address may be an IPv4 dotted address or an IPv6 colon-delimited address. It may also be a hostname, which `dig` will resolve (through the nameservers in `/etc/resolv.conf`).
- `name`: The domain name to look up.
- `type`: The type of query to perform, such as `A`, `ANY`, `MX`, `SIG`, and so forth. The default is `A`, but you may use any valid BIND9
query type.

**FLAGS:**

|Flag|Meaning|
|---|---|
|`@8.8.8.8`|query a specific DNS server directly (put right after `dig`)|
|`A` (default)|the record type — IPv4 address|
|`MX`|mail exchange records — where email for that domain gets routed|
|`NS`|nameserver records — which servers are authoritative for the domain|
|`TXT`|text records — often used for domain verification, SPF/DKIM email policy|
|`+short`|print just the answer, no headers/footers — great for scripting or quick checks|
|`+trace`|walk the *entire* resolution chain from the root servers down — shows you exactly where in the DNS hierarchy something breaks|
|`+noall +answer`|show only the answer section, suppress the rest|

**EXAMPLE:**

In [ ]:
$ dig google.com

; <<>> DiG 9.18.28-1~deb12u2-Debian <<>> google.com
;; global options: +cmd
;; Got answer:
;; ->>HEADER<<- opcode: QUERY, status: NOERROR, id: 4821
;; flags: qr rd ra; QUERY: 1, ANSWER: 1, AUTHORITY: 0, ADDITIONAL: 1

;; QUESTION SECTION:
;google.com.                    IN      A

;; ANSWER SECTION:
google.com.             211     IN      A       142.250.183.14

;; Query time: 18 msec
;; SERVER: 127.0.0.53#53(127.0.0.53) (UDP)
;; WHEN: Tue Sep 29 10:02:11 UTC 2026
;; MSG SIZE  rcvd: 55

**READ THE OUTPUT:**

- **`status: NOERROR`** — the query itself succeeded. This is *not* the same as "the record exists" — check the ANSWER section for that. `status: NXDOMAIN` means the domain doesn't exist at all. `status: SERVFAIL` means the DNS server itself hit an error trying to resolve it (often a sign of a broken/misconfigured nameserver somewhere in the chain).
- **`flags: qr rd ra`** — `qr` = this is a response, `rd` = recursion was desired (client asked the server to do the full lookup chain for it), `ra` = recursion is available (server supports doing that). **Missing `ra` on a server you expect to be a full resolver is a red flag**.
- **`ANSWER: 1`** — how many records came back. `ANSWER: 0` with `status: NOERROR` is a specific, confusing case: the domain exists, but has **no record of the type you asked for** (e.g., you asked for `MX` on a domain that doesn't do email).
- **`ANSWER SECTION`** — `211` is the **TTL** (seconds this record is cached before a resolver must re-ask) — a very short TTL (like 30-60s) is often intentional, set right before a planned IP change/migration/failover, so a short TTL on a domain that's misbehaving is a clue someone is actively changing something.
- **`Query time: 18 msec`** — how long the DNS server itself took to answer. High query time (hundreds of ms) with the app otherwise idle points at a slow/overloaded DNS server, not your network path.
- **`SERVER: 127.0.0.53#53`** — same idea as `nslookup`'s `Server` line: which resolver actually answered.

**TRY NOW:**
```
dig google.com
dig google.com +short
dig google.com MX
dig +trace google.com
```
The `+trace` output is long — that's the point. It shows every nameserver hop: root servers → `.com` TLD servers → google.com's own authoritative servers. If a domain is broken partway through its own DNS chain (like a misconfigured nameserver delegation), `+trace` is what exposes exactly which hop drops the ball — `dig` alone won't show you that, it just gives you the final answer or failure.

**TICKET SCENARIO:** "Users report our new subdomain `app.example.com` isn't loading for anyone, right after we launched it." Before touching the app or load balancer:
```
dig app.example.com +short
```
- **No output at all** → the DNS record was never created, or hasn't propagated yet — this is a DNS/infra ticket, not an app bug.
- **Old/wrong IP returned** → likely a caching issue (check the TTL) or the record was never updated — again, DNS team's problem, not the app's.
- **Correct IP returned** → DNS is fine, now you'd pivot to `curl`/`nc` against that IP to check if the app itself is actually listening and responding.

**DIAGNOSIS PATH:**

1. `dig <host> +short` — does it resolve at all, and to what?
2. If wrong/missing, `dig <host> NS` — check who's actually authoritative for the domain; sometimes the nameservers themselves are misconfigured or point at the wrong provider.
3. `dig @8.8.8.8 <host>` vs `dig <host>` (your default resolver) — if a public resolver gives a different/correct answer than your local one, the problem is caching or misconfiguration on *your* resolver, not the domain's DNS setup.
4. `dig +trace <host>` — walk the full chain when you need to find exactly which nameserver in the hierarchy is misbehaving.
5. Once resolution is confirmed correct, DNS is cleared — move on to `ping`/`traceroute`/`curl` against the resolved IP.

**GOTCHA:** DNS caching means a fix can be correct but *appear* broken for a while — both on DNS resolvers and often in your own browser/OS cache. Always check the record's **TTL** before assuming a fix "didn't work"; a record with a 3600s (1hr) TTL can take up to an hour to update everywhere after a change, and that's expected behavior, not a bug. Also, WSL-specific: WSL2's `/etc/resolv.conf` is often auto-generated and points at a Windows-host-provided DNS server — if DNS behaves differently in WSL than on your Windows host, `cat /etc/resolv.conf` first to see what resolver WSL is actually configured to use.

**TRY NOW (do these in your WSL terminal, then paste real output into the cells below):**
```
nslookup google.com
nslookup google.com 8.8.8.8
dig google.com
dig google.com +short
dig google.com MX
dig +trace google.com
```

In [ ]:
dennie@X390:~/linux-tut$ nslookup google.com
Server:         10.255.255.254
Address:        10.255.255.254#53

Non-authoritative answer:
Name:   google.com
Address: 142.250.120.113
Name:   google.com
Address: 142.250.120.101
Name:   google.com
Address: 142.250.120.138
Name:   google.com
Address: 142.250.120.102
Name:   google.com
Address: 142.250.120.139
Name:   google.com
Address: 142.250.120.100
Name:   google.com
Address: 2a00:1450:4025:800::65
Name:   google.com
Address: 2a00:1450:4025:800::66
Name:   google.com
Address: 2a00:1450:4025:800::8a
Name:   google.com
Address: 2a00:1450:4025:800::8b

In [ ]:
dennie@X390:~/linux-tut$ nslookup google.com 8.8.8.8
Server:         8.8.8.8
Address:        8.8.8.8#53

Non-authoritative answer:
Name:   google.com
Address: 142.250.109.102
Name:   google.com
Address: 142.250.109.100
Name:   google.com
Address: 142.250.109.113
Name:   google.com
Address: 142.250.109.101
Name:   google.com
Address: 142.250.109.138
Name:   google.com
Address: 142.250.109.139
Name:   google.com
Address: 2a00:1450:4025:800::71
Name:   google.com
Address: 2a00:1450:4025:800::8b
Name:   google.com
Address: 2a00:1450:4025:800::66
Name:   google.com
Address: 2a00:1450:4025:800::64

**`nslookup google.com` (default) and `nslookup google.com 8.8.8.8`** — both resolved fine, each returning 6 IPv4 + 4 IPv6 addresses (Google load-balances across many front-end IPs — normal). Different IP sets from the two servers is also completely normal; Google's anycast/CDN returns different front-ends depending on which resolver asks.

**Correction to the lesson's assumption:** notice your default resolver is `10.255.255.254`, not `127.0.0.53` like the lesson's canned example showed. Your `resolvectl status` output explains exactly why: `resolv.conf mode: foreign`. That means systemd-resolved detected that `/etc/resolv.conf` isn't its own managed stub file — WSL is auto-generating it directly (the comment at the top of your `cat /etc/resolv.conf` output confirms this) and pointing it straight at `10.255.255.254`, which is the WSL2 vEthernet's own DNS-forwarding proxy to the Windows host. So in your setup, `dig`/`nslookup` talk directly to that WSL proxy, bypassing the local `127.0.0.53` stub entirely. **Lesson correction: never assume `127.0.0.53` — always check `cat /etc/resolv.conf` first, exactly like you just did.** Good catch material for the notebook.

In [ ]:
dennie@X390:~/linux-tut$ dig google.com

; <<>> DiG 9.18.39-0ubuntu0.24.04.7-Ubuntu <<>> google.com
;; global options: +cmd
;; Got answer:
;; ->>HEADER<<- opcode: QUERY, status: NOERROR, id: 11784
;; flags: qr rd ra; QUERY: 1, ANSWER: 6, AUTHORITY: 0, ADDITIONAL: 1

;; OPT PSEUDOSECTION:
; EDNS: version: 0, flags:; udp: 1232
;; QUESTION SECTION:
;google.com.                    IN      A

;; ANSWER SECTION:
google.com.             68      IN      A       142.251.98.100
google.com.             68      IN      A       142.251.98.102
google.com.             68      IN      A       142.251.98.113
google.com.             68      IN      A       142.251.98.139
google.com.             68      IN      A       142.251.98.138
google.com.             68      IN      A       142.251.98.101

;; Query time: 220 msec
;; SERVER: 10.255.255.254#53(10.255.255.254) (UDP)
;; WHEN: Wed Sep 30 17:23:12 CEST 2026
;; MSG SIZE  rcvd: 135

In [ ]:
dennie@X390:~/linux-tut$ dig google.com +short
142.251.98.138
142.251.98.139
142.251.98.113
142.251.98.102
142.251.98.100
142.251.98.101

In [ ]:
dennie@X390:~/linux-tut$ dig google.com MX

; <<>> DiG 9.18.39-0ubuntu0.24.04.7-Ubuntu <<>> google.com MX
;; global options: +cmd
;; Got answer:
;; ->>HEADER<<- opcode: QUERY, status: NOERROR, id: 39433
;; flags: qr rd ra; QUERY: 1, ANSWER: 1, AUTHORITY: 0, ADDITIONAL: 1

;; OPT PSEUDOSECTION:
; EDNS: version: 0, flags:; udp: 1232
;; QUESTION SECTION:
;google.com.                    IN      MX

;; ANSWER SECTION:
google.com.             300     IN      MX      10 smtp.google.com.

;; Query time: 277 msec
;; SERVER: 10.255.255.254#53(10.255.255.254) (UDP)
;; WHEN: Wed Sep 30 17:25:23 CEST 2026
;; MSG SIZE  rcvd: 60

**`dig google.com`, `+short`, `MX`** — all clean, `status: NOERROR`, consistent IPs, MX record correctly showing `smtp.google.com`. TTL of 68s and 300s respectively — short TTLs, consistent with Google's high-churn load-balanced infrastructure, not a problem.

In [ ]:
dennie@X390:~/linux-tut$ dig +trace google.com
;; communications error to 10.255.255.254#53: timed out
;; communications error to 10.255.255.254#53: timed out
;; communications error to 10.255.255.254#53: timed out

; <<>> DiG 9.18.39-0ubuntu0.24.04.7-Ubuntu <<>> +trace google.com
;; global options: +cmd
;; no servers could be reached

**this one actually failed, and it's a genuinely useful failure**:
Here's why, mechanically: `+trace` doesn't skip your configured resolver entirely — it first sends a **priming query** (asking for the root zone's NS records) _to your configured resolver_ to get the root server list, and only _after_ that does it start talking to nameservers directly. That priming query went to `10.255.255.254` and timed out three times, so the trace never even started.

This is notable because your _regular_ `dig`/`MX` queries to that exact same server succeeded seconds earlier. The likely explanation: the WSL DNS proxy at `10.255.255.254` is a lightweight forwarder built for ordinary recursive lookups — it may not handle (or may silently drop) a raw `NS .` query the way a full recursive resolver like `8.8.8.8` does.
**TICKET SCENARIO this maps to:** "some DNS queries work, but a specific tool/query type fails" — don't conclude "DNS is broken," conclude "this specific resolver doesn't support this specific query." Fix: point the tool at a resolver you know handles it fully.

**Try this now** to confirm the diagnosis:

In [ ]:
dennie@X390:~/linux-tut$ dig +trace google.com @8.8.8.8

; <<>> DiG 9.18.39-0ubuntu0.24.04.7-Ubuntu <<>> +trace google.com @8.8.8.8
;; global options: +cmd
.                       87203   IN      NS      h.root-servers.net.
.                       87203   IN      NS      b.root-servers.net.
.                       87203   IN      NS      a.root-servers.net.
.                       87203   IN      NS      j.root-servers.net.
.                       87203   IN      NS      m.root-servers.net.
.                       87203   IN      NS      k.root-servers.net.
.                       87203   IN      NS      e.root-servers.net.
.                       87203   IN      NS      g.root-servers.net.
.                       87203   IN      NS      l.root-servers.net.
.                       87203   IN      NS      c.root-servers.net.
.                       87203   IN      NS      f.root-servers.net.
.                       87203   IN      NS      i.root-servers.net.
.                       87203   IN      NS      d.root-servers.net.
.                       87203   IN      RRSIG   NS 8 0 518400 20261013120000 20260930110000 57780 . T7vhha8aHMNAEEktBOeInhx4fcDCuMhdxNwQIBo/cxied9Xjwc1d8JeJ Vv64dMwtuqiWOyCnA242UVM4IxM6fT7LDPayuV2eW3plD+5RVyv+xeLC TL/+FDYYflFHqp8f2rF3wOYtRTdXHDznP6ARAaOwZNq2VMSKY4gfrxR5 ir/le7pQE14SPqCWx5TXjt7v2OKNrXGS9V1DdDIsK92oP/0t5DgdPQ3K 7aYxUjULzqrWsbJaZMQ6e/CBvfYcO/UupkBX5/QveI8i2itJm7pZ0dHg p6CKPiRm1rH8ow/gtZvGCrHWwpyZ8tVBGltLWaWGwQtsVNIz557TsJJc MYeqqQ==
;; Received 525 bytes from 8.8.8.8#53(8.8.8.8) in 1009 ms

;; UDP setup with 2001:7fd::1#53(2001:7fd::1) for google.com failed: network unreachable.
;; no servers could be reached
;; UDP setup with 2001:7fd::1#53(2001:7fd::1) for google.com failed: network unreachable.
;; no servers could be reached
;; UDP setup with 2001:7fd::1#53(2001:7fd::1) for google.com failed: network unreachable.
com.                    172800  IN      NS      f.gtld-servers.net.
com.                    172800  IN      NS      e.gtld-servers.net.
com.                    172800  IN      NS      d.gtld-servers.net.
com.                    172800  IN      NS      m.gtld-servers.net.
com.                    172800  IN      NS      h.gtld-servers.net.
com.                    172800  IN      NS      i.gtld-servers.net.
com.                    172800  IN      NS      k.gtld-servers.net.
com.                    172800  IN      NS      g.gtld-servers.net.
com.                    172800  IN      NS      c.gtld-servers.net.
com.                    172800  IN      NS      a.gtld-servers.net.
com.                    172800  IN      NS      b.gtld-servers.net.
com.                    172800  IN      NS      j.gtld-servers.net.
com.                    172800  IN      NS      l.gtld-servers.net.
com.                    86400   IN      DS      19718 13 2 8ACBB0CD28F41250A80A491389424D341522D946B0DA0C0291F2D3D7 71D7805A
com.                    86400   IN      RRSIG   DS 8 1 86400 20261013120000 20260930110000 57780 . kgNKZTatS7T/Nk+9y4HR8bf8ioxxCCoztSIB0C/0astkQ66eHKCiVjNP tumm21oqqftW8GYhnWe6zQgenIZk6uOiMFFj7vT9VI9VqZyQ04hnC0WL LuK1vslJ4Xm6UVmWMP9rug2TU4zXeo0wOtdONHUPA8Sz8cnQO2XhX1OG byTwbPjDg+kH9vd0VwCt7GKeODqs3mc6tax9JMfq3MRjRATctIpZnIvb gpTxPCj8jEeqzRojKTeK7MqUCjvnV3+5OQ8x7NC8M/D+vcl5jonSVVd6 IOWTHGBkgEn5QtvsmHnY7S/riM2SCk+b25hQxYRad/yUgpSuSkWee2MH sY4ZgQ==
;; Received 1198 bytes from 192.112.36.4#53(g.root-servers.net) in 103 ms

;; UDP setup with 2001:502:7094::30#53(2001:502:7094::30) for google.com failed: network unreachable.
google.com.             172800  IN      NS      ns2.google.com.
google.com.             172800  IN      NS      ns1.google.com.
google.com.             172800  IN      NS      ns3.google.com.
google.com.             172800  IN      NS      ns4.google.com.
CK0POJMG874LJREF7EFN8430QVIT8BSM.com. 900 IN NSEC3 1 1 0 - CK0Q35HRS4H76G6CHNB9414CJN6S5UPL NS SOA RRSIG DNSKEY NSEC3PARAM
CK0POJMG874LJREF7EFN8430QVIT8BSM.com. 900 IN RRSIG NSEC3 13 2 900 20261005002614 20260927231614 41446 com. M0YjaiSWHnTC1v/hDEB8FdQWzXfR1eeEDbv9da5DBApuBWBEwSvoWJfJ 5XY5aLsSghKCCbncKx7uwazdiqL1Ww==
S84BOR4DK28HNHPLC218O483VOOOD5D8.com. 900 IN NSEC3 1 1 0 - S84BR9CIB2A20L3ETR1M2415ENPP99L8 NS DS RRSIG
S84BOR4DK28HNHPLC218O483VOOOD5D8.com. 900 IN RRSIG NSEC3 13 2 900 20261006012207 20260929001207 41446 com. zPf9vYpD6UkDU+M/AsbKjBxl0Jx7v5E/7JLCMd0Mp2ngBuVbqI82p4vl CLEIwjqYxYPB8bC14EpVx21GIAZa/Q==
;; Received 644 bytes from 192.26.92.30#53(c.gtld-servers.net) in 667 ms

;; UDP setup with 2001:4860:4802:32::a#53(2001:4860:4802:32::a) for google.com failed: network unreachable.
google.com.             300     IN      A       142.250.109.139
google.com.             300     IN      A       142.250.109.101
google.com.             300     IN      A       142.250.109.113
google.com.             300     IN      A       142.250.109.102
google.com.             300     IN      A       142.250.109.138
google.com.             300     IN      A       142.250.109.100
;; Received 135 bytes from 216.239.34.10#53(ns2.google.com) in 177 ms

That confirms the diagnosis exactly. Full breakdown:

**Root cause confirmed:** targeting `@8.8.8.8` fixed it — the trace ran to completion: root servers → `.com` TLD servers → `google.com`'s own nameservers → final `A` records. This proves the original `+trace` failure was specifically the WSL DNS proxy (`10.255.255.254`) choking on the priming `NS .` query, not a real DNS/network problem.

**One more genuine WSL quirk visible in this output** — the repeated lines:

;; UDP setup with 2001:7fd::1#53 for google.com failed: network unreachable.

At each level of the trace, `dig` picks one nameserver at random from the list it just received to query next — and several of the ones it picked here are **IPv6-only** addresses (`2001:...`). WSL2 commonly has no usable IPv6 route out to the internet, so any IPv6-only nameserver `dig` happens to pick fails instantly with "network unreachable." That's expected and harmless _as long as `dig` eventually retries with an IPv4 nameserver from the same list and succeeds_ — which it did every time here (`192.112.36.4`, `192.26.92.30`, `216.239.34.10` all IPv4, all answered fine). If you ever see `+trace` actually **stop dead** on one of these instead of recovering, that's the point where it'd become a real problem.

**Reading the DNSSEC noise** (the `RRSIG`, `DS`, `NSEC3` lines) — you don't need to understand these deeply for support work, just recognize them as signature/validation records that ride along with modern DNS responses (DNSSEC = cryptographic proof the record wasn't tampered with in transit). They're normal clutter in a `+trace`, not something to diagnose.

**Support takeaway:** two separate WSL-specific behaviors surfaced in one exercise — (1) the local DNS proxy can silently fail on certain query types even while handling normal lookups fine, and (2) `dig +trace` can throw scary-looking "network unreachable" lines that are just IPv6-route gaps, self-correcting on retry. Both are "don't panic, verify before escalating" cases.

**Follow-up Q&A — what "DNS server" actually means, and where it lives on your machine**

Your instinct was right that the resolver "fully interacts with DNS servers along the way" — but **"DNS server"** is actually hiding two very different roles, and untangling them clears up a lot:

1. **Recursive resolver** — the server your machine actually **asks**. It does the legwork (or delegates it) and hands back a final answer. `8.8.8.8` (Google) or `1.1.1.1` (Cloudflare) are this.
2. **Authoritative nameserver** — the server that actually *owns* the answer for a specific domain (e.g., Google's own nameservers hold the real record for `google.com`). Thousands of these exist, one set per domain. Your recursive resolver talks to these on your behalf — you never talk to them directly, except with `dig +trace`, which is exactly why that flag exists: it exposes that normally-hidden conversation.

**Where this lives on your machine — concretely:**
```
cat /etc/resolv.conf
resolvectl status
```

In [ ]:
dennie@X390:~/linux-tut$ cat /etc/resolv.conf
# This file was automatically generated by WSL. To stop automatic generation of this file, add the following entry to /etc/wsl.conf:
# [network]
# generateResolvConf = false
nameserver 10.255.255.254

`cat /etc/resolv.conf` shows the IP your system currently sends DNS queries to. 

In [ ]:
dennie@X390:~/linux-tut$ resolvectl status
Global
         Protocols: -LLMNR -mDNS -DNSOverTLS DNSSEC=no/unsupported
  resolv.conf mode: foreign
Current DNS Server: 10.255.255.254
       DNS Servers: 10.255.255.254

Link 2 (eth0)
    Current Scopes: none
         Protocols: -DefaultRoute -LLMNR -mDNS -DNSOverTLS DNSSEC=no/unsupported

`resolvectl status` (systemd-resolved — this is what `127.0.0.53` in the `dig`/`nslookup` output actually is) shows the *real* upstream resolver it's forwarding to behind that local stub address.

**The full chain:**
```shell
Browser / curl / dig
        |
        v
Stub resolver (local, on YOUR machine)
  - Linux/WSL: systemd-resolved at 127.0.0.53, config in /etc/resolv.conf
  - Windows: OS-level DNS client service
  - Caches recent answers, forwards new queries upward
        |
        v
Your CONFIGURED recursive resolver
  - Usually handed to you automatically by DHCP from your ISP or router
  - Or manually overridden to 8.8.8.8, 1.1.1.1, etc.
  - THIS is the "DNS server" nslookup/dig call out by IP in their output
        |
        v  (only if not cached -- this is the actual recursive walk)
Root servers -> TLD servers (.com, .org...) -> Authoritative nameservers
  for the specific domain
```

So "your configured DNS server is the problem" means **the recursive resolver your machine happens to be pointed at** is down, has a stale/bad cache entry, or is misconfigured -- while the domain's actual records (the authoritative layer) are completely fine. That's exactly why `dig google.com` vs `dig google.com 8.8.8.8` is a real diagnostic:
> if the two disagree, the fault is isolated to *your* resolver, not the domain.

**Wrinkle worth knowing:** modern browsers (Chrome, Firefox) increasingly do **DNS over HTTPS (DoH)** and may bypass your OS's configured resolver entirely, talking straight to their own built-in resolver (often Cloudflare). This is why a domain can occasionally resolve differently -- or work -- in a browser but fail in `dig`/`curl`/`ping` on the same machine: **they may literally not be asking the same resolver**. If a ticket says "works in the browser but our script/curl can't reach it," a DNS-path mismatch between the browser's resolver and the OS's resolver is a real thing to check for.

**Practical takeaway for support work:** when DNS looks broken, you're really asking "which of the four layers above is lying to me?" -- and `dig @<server> host` is the tool that pins the fault to one specific hop instead of guessing.